# Lenguajes y Autómatas II (SCD-1016)
## Sesión 02 · Gestión en GitHub Web y Nodos Dinámicos en Heap

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo guarda un compilador una cantidad de datos que no conoce de antemano?
>
> **Objetivo de la sesión:** Construir una lista enlazada de tokens en el heap, reservando y liberando memoria nodo por nodo sin fugas ni accesos a memoria liberada.

**Entregable:** tu lista enlazada en artesano_sesion02_c.c, el harness en verde y la bitácora de la sesión

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes la cabeza de una lista y un lexema, y devuelves la lista con el token agregado al final. El harness comprueba el orden, el final de la cadena y que liberar no reviente.

Al final de la sesión tendrás: **Una lista enlazada de tokens que crece token por token, se cuenta, se recorre y se libera sin fugas.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 02](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion02.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion02.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion02.h"

NodoToken *agregar_token(NodoToken *cabeza, const char *lexema);
int contar_tokens(NodoToken *cabeza);
void liberar_lista(NodoToken *cabeza);
```

- Cada nodo se reserva con malloc; el campo siguiente del último vale NULL.
- agregar_token devuelve la cabeza: si la lista estaba vacía, la nueva cabeza es el nodo creado.
- liberar_lista(NULL) no debe caerse.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion02_c.c` junto con `artesano_sesion02_c.c`
> (o `ia_sesion02_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `agregar_token(NULL,"if")` | `if` |
| 2 | `contar_tokens(3 tokens)` | 3 |
| 3 | `orden: segundo token` | `dos` |
| 4 | `ultimo->siguiente == NULL` | 1 |
| 5 | `contar_tokens(NULL)` | 0 |
| 6 | `liberar_lista(NULL)` | `sin crash` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 02 - Lista enlazada de tokens en heap ===
Caso                               esperado:              obtenido:              resultado
[ 1] agregar_token(NULL,"if")           esperado: if           obtenido: if           OK
[ 2] contar_tokens(3 tokens)            esperado: 3            obtenido: 3            OK
[ 3] orden: segundo token               esperado: dos          obtenido: dos          OK
[ 4] ultimo->siguiente == NULL          esperado: 1            obtenido: 1            OK
[ 5] contar_tokens(NULL)                esperado: 0            obtenido: 0            OK
[ 6] liberar_lista(NULL)                esperado: sin crash    obtenido: sin crash    OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion02.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Cada token ocupa un nodo reservado con malloc y encadenado con siguiente.
- [ ] agregar_token devuelve la lista completa aunque la cabeza cambie.
- [ ] contar_tokens(NULL) devuelve 0 y liberar_lista(NULL) no se cae.


## Paso 0 · Entorno


In [24]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion02_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [25]:
%%writefile api_sesion02.h
/* api_sesion02.h - contrato de la Sesion 02
 * Lista enlazada de tokens en heap
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION02_H
#define API_SESION02_H

typedef struct NodoToken {
    char lexema[16];
    struct NodoToken *siguiente;
} NodoToken;

NodoToken *agregar_token(NodoToken *cabeza, const char *lexema);
int contar_tokens(NodoToken *cabeza);
void liberar_lista(NodoToken *cabeza);

#endif /* API_SESION02_H */


Overwriting api_sesion02.h


In [26]:
%%writefile artesano_sesion02_c.c
/* artesano_sesion02_c.c - Fase 1: El Artesano (sin IA)
 * Lista enlazada de tokens en heap.
 */
#include "api_sesion02.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

NodoToken *agregar_token(NodoToken *cabeza, const char *lexema) {
    /* si no me dan lexema, dejo la lista como estaba */
    if (lexema == NULL) {
        return cabeza;
    }

    /* reservo el nodo nuevo en el heap */
    NodoToken *nuevo = malloc(sizeof(NodoToken));
    if (nuevo == NULL) {
        return cabeza; /* si malloc falla, no se agrega nada */
    }

    /* copio el lexema sin pasarme de los 16 bytes del arreglo */
    strncpy(nuevo->lexema, lexema, sizeof(nuevo->lexema) - 1);
    nuevo->lexema[sizeof(nuevo->lexema) - 1] = '\0'; /* aseguro el fin de cadena */
    nuevo->siguiente = NULL;

    /* si la lista estaba vacia, el nodo nuevo es la cabeza */
    if (cabeza == NULL) {
        return nuevo;
    }

    /* si no, camino hasta el ultimo nodo y lo enlazo ahi */
    NodoToken *actual = cabeza;
    while (actual->siguiente != NULL) {
        actual = actual->siguiente;
    }
    actual->siguiente = nuevo;

    return cabeza;
}

int contar_tokens(NodoToken *cabeza) {
    int cuenta = 0;
    NodoToken *actual = cabeza;

    /* recorro la lista sumando uno por cada nodo */
    while (actual != NULL) {
        cuenta++;
        actual = actual->siguiente;
    }
    return cuenta;
}

void liberar_lista(NodoToken *cabeza) {
    NodoToken *actual = cabeza;
    NodoToken *siguiente;

    while (actual != NULL) {
        siguiente = actual->siguiente; /* guardo el siguiente ANTES de liberar */
        free(actual);
        actual = siguiente;
    }
}


Overwriting artesano_sesion02_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion02_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


Contexto: Estoy en la materia Lenguajes y Autómatas II. Trabajo en C (estándar C11) y compilo con gcc -std=c11 -Wall -Wextra -Werror, así que cualquier advertencia cuenta como error. Ya existe un archivo api_sesion02.h con esta estructura y estas firmas:
typedef struct NodoToken { char lexema[16]; struct NodoToken *siguiente; } NodoToken;
NodoToken *agregar_token(NodoToken *cabeza, const char *lexema);
int contar_tokens(NodoToken *cabeza);
void liberar_lista(NodoToken *cabeza);

Rol: Actúa como ingeniero de software de sistemas y compiladores, con experiencia en memoria dinámica y listas enlazadas en C.

Especificación:
- agregar_token reserva un nodo con malloc, copia el lexema sin desbordar el arreglo de 16 bytes y lo agrega al final. Devuelve la cabeza de la lista (si estaba vacía, devuelve el nodo nuevo).
- contar_tokens devuelve cuántos nodos tiene la lista, y 0 si la cabeza es NULL.
- liberar_lista libera todos los nodos sin usar memoria ya liberada, y no debe caerse con NULL.
- El campo siguiente del último nodo debe valer NULL.
- No uses main() ni cambies las firmas ni la estructura.

Output: Dame solo el código del archivo ia_sesion02_c.c con comentarios breves, y al final un análisis corto de la complejidad en tiempo y en espacio.

In [27]:
%%writefile ia_sesion02_c.c
/* ia_sesion02_c.c - Fase 2: El Cientifico
 * Propuesta de la IA, revisada y corregida por mi.
 */
#include "api_sesion02.h"
#include <stdio.h>
#include <stdlib.h>

NodoToken *agregar_token(NodoToken *cabeza, const char *lexema) {
    if (lexema == NULL) {
        return cabeza;
    }

    NodoToken *nuevo = malloc(sizeof(NodoToken));
    if (nuevo == NULL) {
        return cabeza;
    }

    /* snprintf corta el texto a 15 caracteres y siempre pone el '\0' */
    snprintf(nuevo->lexema, sizeof(nuevo->lexema), "%s", lexema);
    nuevo->siguiente = NULL;

    if (cabeza == NULL) {
        return nuevo;
    }

    NodoToken *ultimo = cabeza;
    while (ultimo->siguiente != NULL) {
        ultimo = ultimo->siguiente;
    }
    ultimo->siguiente = nuevo;
    return cabeza;
}

int contar_tokens(NodoToken *cabeza) {
    int total = 0;
    while (cabeza != NULL) {
        total++;
        cabeza = cabeza->siguiente;
    }
    return total;
}

void liberar_lista(NodoToken *cabeza) {
    while (cabeza != NULL) {
        NodoToken *borrar = cabeza;
        cabeza = cabeza->siguiente; /* avanzo antes de liberar el nodo */
        free(borrar);
    }
}

/* Complejidad: agregar_token es O(n) porque recorre la lista hasta el final.
   contar_tokens y liberar_lista son O(n). El espacio extra es O(1) en las
   tres funciones, aparte del nodo que se reserva en agregar_token. */


Overwriting ia_sesion02_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [28]:
%%writefile harness_sesion02_c.c
/* harness_sesion02_c.c - auditoria automatica de la Sesion 02
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion02_c.c artesano_sesion02_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion02.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 02 - Lista enlazada de tokens en heap ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { NodoToken *c = agregar_token(NULL, "if"); verificar_str("agregar_token(NULL,\"if\")", "if", (c != NULL ? c->lexema : "(NULL)")); }
    { NodoToken *c = NULL; c = agregar_token(c, "if"); c = agregar_token(c, "x"); c = agregar_token(c, ";"); verificar_int("contar_tokens(3 tokens)", (long)(3), (long)(contar_tokens(c))); }
    { NodoToken *c = NULL; c = agregar_token(c, "uno"); c = agregar_token(c, "dos"); verificar_str("orden: segundo token", "dos", (c && c->siguiente ? c->siguiente->lexema : "(NULL)")); }
    { NodoToken *c = NULL; c = agregar_token(c, "a"); c = agregar_token(c, "b"); verificar_int("ultimo->siguiente == NULL", (long)(1), (long)((c && c->siguiente ? (c->siguiente->siguiente == NULL) : 0))); }
    {  verificar_int("contar_tokens(NULL)", (long)(0), (long)(contar_tokens(NULL))); }
    { NodoToken *c = NULL; c = agregar_token(c, "a"); liberar_lista(c); liberar_lista(NULL); verificar_str("liberar_lista(NULL)", "sin crash", "sin crash"); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion02.md" "\n");
    return 0;
}


Overwriting harness_sesion02_c.c


In [29]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion02_c.c artesano_sesion02_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 02 - Lista enlazada de tokens en heap ===
Caso                               esperado:              obtenido:              resultado
[ 1] agregar_token(NULL,"if")           esperado: if           obtenido: if           OK
[ 2] contar_tokens(3 tokens)            esperado: 3            obtenido: 3            OK
[ 3] orden: segundo token               esperado: dos          obtenido: dos          OK
[ 4] ultimo->siguiente == NULL          esperado: 1            obtenido: 1            OK
[ 5] contar_tokens(NULL)                esperado: 0            obtenido: 0            OK
[ 6] liberar_lista(NULL)                esperado: sin crash    obtenido: sin crash    OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion02.md


In [30]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion02_c.c ia_sesion02_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 02 - Lista enlazada de tokens en heap ===
Caso                               esperado:              obtenido:              resultado
[ 1] agregar_token(NULL,"if")           esperado: if           obtenido: if           OK
[ 2] contar_tokens(3 tokens)            esperado: 3            obtenido: 3            OK
[ 3] orden: segundo token               esperado: dos          obtenido: dos          OK
[ 4] ultimo->siguiente == NULL          esperado: 1            obtenido: 1            OK
[ 5] contar_tokens(NULL)                esperado: 0            obtenido: 0            OK
[ 6] liberar_lista(NULL)                esperado: sin crash    obtenido: sin crash    OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion02.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion02.md` a `auditorias/auditoria_sesion02.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
